# Benchmarking Network Inference Methods on Transient Data

This notebook compares multiple methods on transient biological time-series data.


In [3]:
import numpy as np
import graph_tool.all as gt
import sys
import os
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, roc_curve, precision_recall_curve, auc
from scipy.stats import spearmanr



# Add the parent directory to sys.path so we can import our modules and gene_expr

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

from inference_pipeline.data_generation.network_generator import generate_scale_free_W
from inference_pipeline.data_generation.simulator import simulate_realistic_pairs
from inference_pipeline.inference.mcmc_engine import run_burn_in, run_posterior_sampling, compute_ensemble_stats
from inference_pipeline.inference.metrics import plot_probabilistic_network, plot_inferred_vs_true_am, compute_reconstruction_metrics
from inference_pipeline.prediction.export_utils import export_ensemble_to_bin, export_true_to_bin
from gene_expr import GeneExprBlockState


In [1]:
N = 10
m = 1
seeds_network = [45, 100, 2023] # 3 different network topologies
dt_vals = [0.1, 0.5, 1.0, 2.0] # 5 points for dt

dt_sim = 0.005
M_train = 50
M_test = 50
t_window = (0.0, 6.0)

mean_d = 0.5
mean_V = 2.0
D_base = 0.005

In [6]:
import pandas as pd
from tqdm.notebook import tqdm
import time
import os

def calc_all_metrics(y_true_binary, y_true_continuous, y_scores, y_inferred_continuous=None):
    mask = ~np.eye(N, dtype=bool)
    y_true_b = y_true_binary[mask]
    y_s = y_scores[mask]
    
    fpr, tpr, _ = roc_curve(y_true_b, y_s)
    roc_auc = auc(fpr, tpr)
    
    precision, recall, _ = precision_recall_curve(y_true_b, y_s)
    pr_auc = auc(recall, precision)
    
    spearman_corr, _ = spearmanr(np.abs(y_true_continuous[mask]), np.abs(y_s))
    
    sign_acc = np.nan
    if y_inferred_continuous is not None:
        true_edges_mask = (y_true_b == 1)
        if np.any(true_edges_mask):
            true_signs = np.sign(y_true_continuous[mask][true_edges_mask])
            inf_signs = np.sign(y_inferred_continuous[mask][true_edges_mask])
            sign_acc = np.mean(true_signs == inf_signs)
    
    return fpr, tpr, roc_auc, precision, recall, pr_auc, spearman_corr, sign_acc

results_list = []

for seed_network in tqdm(seeds_network, desc="Network Seeds"):
    # Generate TRUE network
    W_true = generate_scale_free_W(N, m, seed=seed_network)
    W_true_binary = (W_true != 0).astype(int)
    theta_true = [np.log(mean_d), np.log(mean_V), np.log(D_base)]
    
    for dt_meas in tqdm(dt_vals, desc=f"dt values (seed {seed_network})", leave=False):
        dt_sample = dt_meas
        
        # Simulate Training Data
        s_train, _, _, _, _ = simulate_realistic_pairs(
            W_true, M=M_train, dt_sim=dt_sim, dt_sample=dt_sample, dt_meas=dt_meas, 
            D_base=D_base, mean_d=mean_d, std_d=0.0, mean_V=mean_V, std_V=0.0, seed=100+seed_network, t_window=t_window
        )
        
        # --- Standard Likelihood ---
        g_init = gt.Graph(directed=True)
        g_init.add_vertex(N)
        num_initial_edges = max(1, int(N * N * 0.05))
        possible_edges = [(u, v) for u in range(N) for v in range(N) if u != v]
        np.random.seed(42)
        np.random.shuffle(possible_edges)
        for u, v in possible_edges[:num_initial_edges]:
            g_init.add_edge(u, v)

        initial_theta = [0.0, 0.0, np.log(0.05)]
        
        # State without EM
        state_std = GeneExprBlockState(s_train, dt=dt_meas, g=g_init.copy(), state_args=dict(deg_corr=False), theta=initial_theta.copy(), use_em_likelihood=False)
        state_std = run_burn_in(state_std, n_burnin=50, theta_mode="global")
        W_ensemble_std, _ = run_posterior_sampling(state_std, N, n_samples=100, delta_k=1, theta_mode="global")
        edge_prob_std, weight_mean_std, _ = compute_ensemble_stats(W_ensemble_std)
        scores_std = edge_prob_std * np.abs(weight_mean_std)
        _, _, roc_auc_std, _, _, pr_auc_std, spearman_std, _ = calc_all_metrics(W_true_binary, W_true, scores_std, weight_mean_std)
        
        results_list.append({
            'Seed': seed_network,
            'dt': dt_meas,
            'Method': 'Standard',
            'AUROC': roc_auc_std,
            'AUPRC': pr_auc_std,
            'Spearman': spearman_std
        })
        
        # --- EM Likelihood ---
        state_em = GeneExprBlockState(s_train, dt=dt_meas, g=g_init.copy(), state_args=dict(deg_corr=False), theta=initial_theta.copy(), use_em_likelihood=True)
        state_em = run_burn_in(state_em, n_burnin=50, theta_mode="global")
        W_ensemble_em, _ = run_posterior_sampling(state_em, N, n_samples=100, delta_k=1, theta_mode="global")
        edge_prob_em, weight_mean_em, _ = compute_ensemble_stats(W_ensemble_em)
        scores_em = edge_prob_em * np.abs(weight_mean_em)
        _, _, roc_auc_em, _, _, pr_auc_em, spearman_em, _ = calc_all_metrics(W_true_binary, W_true, scores_em, weight_mean_em)
        
        results_list.append({
            'Seed': seed_network,
            'dt': dt_meas,
            'Method': 'EM',
            'AUROC': roc_auc_em,
            'AUPRC': pr_auc_em,
            'Spearman': spearman_em
        })

# Save results
os.makedirs("data", exist_ok=True)
df_results = pd.DataFrame(results_list)
df_results.to_csv("data/likelihood_benchmark_results.csv", index=False)
display(df_results.head())

Network Seeds:   0%|          | 0/3 [00:00<?, ?it/s]

dt values (seed 45):   0%|          | 0/4 [00:00<?, ?it/s]


--- PHASE 1: Equilibration (50 sweeps) ---
Burn-in Sweep  1/50: Entropy =  37.7029 (dS_topo = -147.5350, dS_param = -168.2260, moves = 7513)
Burn-in Sweep 10/50: Entropy = -117.7022 (dS_topo =  21.7262, dS_param =  -2.5873, moves = 9412)
Burn-in Sweep 20/50: Entropy = -68.0235 (dS_topo =   6.4393, dS_param =  -0.0316, moves = 6601)
Burn-in Sweep 30/50: Entropy = -152.4526 (dS_topo =  19.6272, dS_param =  -1.5713, moves = 6213)
Burn-in Sweep 40/50: Entropy = -84.9803 (dS_topo =  43.9286, dS_param =  -0.3007, moves = 5490)
Burn-in Sweep 50/50: Entropy = -268.8373 (dS_topo =   2.2907, dS_param =  -0.9051, moves = 8405)

--- PHASE 2: Sampling (100 samples, thinning 1) ---
Sample   1/100: Entropy = -268.9675
Sample  10/100: Entropy = -230.4572
Sample  20/100: Entropy = -127.8975
Sample  30/100: Entropy = -159.2637
Sample  40/100: Entropy = -204.7711
Sample  50/100: Entropy = -207.4761
Sample  60/100: Entropy = -200.0149
Sample  70/100: Entropy = -152.7103
Sample  80/100: Entropy =  17.5685

dt values (seed 100):   0%|          | 0/4 [00:00<?, ?it/s]


--- PHASE 1: Equilibration (50 sweeps) ---
Burn-in Sweep  1/50: Entropy = 247.9661 (dS_topo = -160.8561, dS_param = -24.7177, moves = 6878)
Burn-in Sweep 10/50: Entropy = -172.8920 (dS_topo =  41.4949, dS_param =  -0.5638, moves = 10072)
Burn-in Sweep 20/50: Entropy = -189.1028 (dS_topo =  -3.8258, dS_param =  -0.0143, moves = 6585)
Burn-in Sweep 30/50: Entropy = -221.6146 (dS_topo =   4.3575, dS_param =  -0.7614, moves = 6360)
Burn-in Sweep 40/50: Entropy = -202.9000 (dS_topo =  10.2697, dS_param =  -0.1026, moves = 7188)
Burn-in Sweep 50/50: Entropy = -199.2184 (dS_topo = -27.9200, dS_param =  -0.0522, moves = 9172)

--- PHASE 2: Sampling (100 samples, thinning 1) ---
Sample   1/100: Entropy = -190.6555
Sample  10/100: Entropy = -178.6366
Sample  20/100: Entropy = -129.6926
Sample  30/100: Entropy = -116.9598
Sample  40/100: Entropy = -67.9854
Sample  50/100: Entropy = -65.8454
Sample  60/100: Entropy = -84.4095
Sample  70/100: Entropy = -63.2460
Sample  80/100: Entropy = -18.8699
S

dt values (seed 2023):   0%|          | 0/4 [00:00<?, ?it/s]


--- PHASE 1: Equilibration (50 sweeps) ---
Burn-in Sweep  1/50: Entropy =  23.5596 (dS_topo = -123.5454, dS_param = -103.9854, moves = 6899)
Burn-in Sweep 10/50: Entropy = -127.4156 (dS_topo =  24.1007, dS_param =  -1.3280, moves = 9076)
Burn-in Sweep 20/50: Entropy = -91.1187 (dS_topo =  36.6516, dS_param =  -0.8651, moves = 5687)
Burn-in Sweep 30/50: Entropy = -138.3268 (dS_topo =  -3.2834, dS_param =  -0.4535, moves = 6046)
Burn-in Sweep 40/50: Entropy = -171.2064 (dS_topo =  -8.4177, dS_param =  -0.5150, moves = 7044)
Burn-in Sweep 50/50: Entropy = -278.8363 (dS_topo =  26.2853, dS_param =  -0.2093, moves = 10878)

--- PHASE 2: Sampling (100 samples, thinning 1) ---
Sample   1/100: Entropy = -282.8414
Sample  10/100: Entropy = -299.6066
Sample  20/100: Entropy = -308.6531
Sample  30/100: Entropy = -299.9831
Sample  40/100: Entropy = -307.8208
Sample  50/100: Entropy = -304.0218
Sample  60/100: Entropy = -317.1340
Sample  70/100: Entropy = -304.3486
Sample  80/100: Entropy = -316.3

,Seed,dt,Method,AUROC,AUPRC,Spearman
0,45,0.1,Standard,0.669410,0.208723,0.185138
1,45,0.1,EM,0.799726,0.457193,0.311915
2,45,0.5,Standard,0.897119,0.832809,0.420606
3,45,0.5,EM,0.853224,0.457790,0.379161
4,45,1.0,Standard,0.866941,0.764183,0.393598


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/likelihood_benchmark_results.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.lineplot(data=df, x='dt', y='AUROC', hue='Method', marker='o', ax=axes[0])
axes[0].set_title('AUROC vs dt')
axes[0].set_xlabel('dt (Sampling & Measurement)')
axes[0].set_ylabel('AUROC')

sns.lineplot(data=df, x='dt', y='AUPRC', hue='Method', marker='o', ax=axes[1])
axes[1].set_title('AUPRC vs dt')
axes[1].set_xlabel('dt (Sampling & Measurement)')
axes[1].set_ylabel('AUPRC')

plt.tight_layout()
plt.show()